# Artifact Rejection — CalmSync Artifact Dataset
## Eye Blinks, Head Nodding & Jaw Clenching
---

This notebook applies multiple artifact removal techniques to **real CalmSync EEG recordings** containing three distinct artifact types:

| Artifact | Source | EEG Signature |
|----------|--------|----------------|
| **Eye Blinks** | Ocular muscles | Large, slow deflections (~0.5–2 Hz) concentrated at frontal electrodes |
| **Head Nodding** | Body movement + electrode shift | Low-frequency drift with abrupt baseline shifts |
| **Jaw Clenching** | EMG from masseter/temporalis | High-frequency broadband burst (>20 Hz) |

We apply **five decomposition methods** and compare their ability to separate artifacts from neural signal:
1. **Wavelet Transform (WT)** — soft thresholding
2. **EMD** — Empirical Mode Decomposition
3. **EEMD** — Ensemble EMD
4. **CEEMDAN** — Complete Ensemble EMD with Adaptive Noise
5. **VMD** — Variational Mode Decomposition

## 1. Load & Preprocess
---
All three recordings come from the same CalmSync device (2-channel, ~240 Hz). We:
- Convert ADC values to **µV**
- Apply a **bandpass filter** (0.5–45 Hz) to remove DC drift and high-frequency noise
- Apply a **50 Hz notch filter** to remove line noise
- Extract the **Fp1** channel (channel 0)

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import pywt
from scipy import signal
from scipy.signal import welch
from PyEMD import EMD, EEMD, CEEMDAN
from vmdpy import VMD

# ── Filter helper ────────────────────────────────────────────────────────────
def preprocess(x, fs, hp=0.5, lp=45.0, notch=50.0):
    """Bandpass + 50 Hz notch filter, then DC removal."""
    sos_bp = signal.butter(4, [hp, lp], btype='band', fs=fs, output='sos')
    x = signal.sosfiltfilt(sos_bp, x)
    b, a = signal.iirnotch(notch, Q=30, fs=fs)
    x = signal.filtfilt(b, a, x)
    x -= x.mean()
    return x

def load_calmsync(path):
    """Load CalmSync JSON, convert to µV, preprocess."""
    with open(path) as f:
        d = json.load(f)
    fs = d['samplingRateHz']
    raw = np.array(d['data'], dtype=float)
    # ADC → µV
    fp1 = ((raw[0] - 512) * 3.3 / 1024) / 1100 * 1e6
    fp1 = preprocess(fp1, fs)
    t = np.arange(len(fp1)) / fs
    return fp1, t, fs

# ── Load all three artifact types ────────────────────────────────────────────
base = r'../calmsync_artifactdata'

blink_sig, blink_t, fs = load_calmsync(f'{base}/eyeblinks/eeg_2026-01-14T17-07-28_2026-01-14T17-08-27.json')
nod_sig, nod_t, _      = load_calmsync(f'{base}/Headnodding/eeg_2026-01-14T17-14-40_2026-01-14T17-15-41.json')
jaw_sig, jaw_t, _      = load_calmsync(f'{base}/jawclenching/eeg_2026-01-14T17-11-51_2026-01-14T17-12-54.json')

artifacts = {
    'Eye Blinks':    (blink_sig, blink_t),
    'Head Nodding':  (nod_sig, nod_t),
    'Jaw Clenching': (jaw_sig, jaw_t),
}

print(f'Sampling rate: {fs:.1f} Hz')
for name, (sig, t) in artifacts.items():
    print(f'{name}: {len(sig)} samples, {t[-1]:.1f}s, '
          f'range: [{sig.min():.1f}, {sig.max():.1f}] µV')

## 2. Raw EEG — All Three Artifacts
---
Visual comparison of the three artifact types. Notice the distinct morphologies:
- **Eye blinks**: sharp, high-amplitude spikes
- **Head nodding**: slow baseline wandering
- **Jaw clenching**: dense high-frequency bursts

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=False)

colors = {'Eye Blinks': '#e74c3c', 'Head Nodding': '#2ecc71', 'Jaw Clenching': '#9b59b6'}

for ax, (name, (sig, t)) in zip(axes, artifacts.items()):
    ax.plot(t, sig, color=colors[name], linewidth=0.5)
    ax.set_title(f'{name} — Fp1', fontsize=12, fontweight='bold')
    ax.set_ylabel('µV')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw Filtered EEG — Three Artifact Types (CalmSync Fp1)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Wavelet Transform (Soft Thresholding)
---
We decompose with the **db4** wavelet (5 levels), estimate noise from the finest detail coefficients using the **MAD estimator**, and apply **soft thresholding**. The denoised signal is then reconstructed.

In [ ]:
def wavelet_denoise(sig, wavelet='db4', level=5):
    """Wavelet soft thresholding using universal threshold (VisuShrink)."""
    coeffs = pywt.wavedec(sig, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(sig)))
    coeffs_thresh = [coeffs[0]] + [
        pywt.threshold(c, value=threshold, mode='soft') for c in coeffs[1:]
    ]
    return pywt.waverec(coeffs_thresh, wavelet)[:len(sig)]

# Apply WT to all three
wt_results = {}
for name, (sig, t) in artifacts.items():
    wt_results[name] = wavelet_denoise(sig)

# Plot original vs denoised
fig, axes = plt.subplots(3, 1, figsize=(14, 8))
for ax, (name, (sig, t)) in zip(axes, artifacts.items()):
    ax.plot(t, sig, color='gray', linewidth=0.4, alpha=0.6, label='Original')
    ax.plot(t, wt_results[name], color=colors[name], linewidth=0.8, label='WT denoised')
    ax.set_title(f'{name} — Wavelet Denoised', fontsize=11)
    ax.set_ylabel('µV')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Wavelet Transform Soft Thresholding (db4, level=5)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. EMD — Empirical Mode Decomposition
---
EMD decomposes each signal into **Intrinsic Mode Functions (IMFs)**. We reconstruct by **removing the first IMF** (highest frequency, typically noise/artifact) and summing the rest.

In [ ]:
def emd_denoise(sig, t, n_remove=1):
    """EMD decomposition, reconstruct by removing first n_remove IMFs."""
    emd = EMD()
    imfs = emd.emd(sig, t)
    reconstructed = np.sum(imfs[n_remove:], axis=0)
    return imfs, reconstructed

emd_results = {}
emd_imfs_all = {}
for name, (sig, t) in artifacts.items():
    imfs, recon = emd_denoise(sig, t)
    emd_imfs_all[name] = imfs
    emd_results[name] = recon
    print(f'{name}: {imfs.shape[0]} IMFs extracted')

# Plot IMFs for each artifact
for name, (sig, t) in artifacts.items():
    imfs = emd_imfs_all[name]
    n_imfs = min(imfs.shape[0], 8)  # show max 8
    fig, axes = plt.subplots(n_imfs + 1, 1, figsize=(14, 1.8 * (n_imfs + 1)), sharex=True)

    axes[0].plot(t, sig, color=colors[name], linewidth=0.5)
    axes[0].set_title(f'Original — {name}', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('µV')

    for i in range(n_imfs):
        f_psd, psd = welch(imfs[i], fs=fs, nperseg=min(512, len(imfs[i])))
        dom_freq = f_psd[np.argmax(psd)]
        energy_pct = np.sum(imfs[i]**2) / np.sum(sig**2) * 100
        axes[i+1].plot(t, imfs[i], color=plt.cm.tab10(i), linewidth=0.5)
        axes[i+1].set_title(f'IMF {i+1}  |  {dom_freq:.1f} Hz  |  {energy_pct:.1f}%',
                            fontsize=9)
        axes[i+1].set_ylabel('µV', fontsize=8)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'EMD — {name}', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 5. EEMD — Ensemble EMD
---
EEMD adds white noise across multiple trials and averages the IMFs to reduce **mode mixing**. We use 50 trials with noise width 0.1.

In [ ]:
def eemd_denoise(sig, t, n_remove=1):
    """EEMD decomposition, reconstruct by removing first n_remove IMFs."""
    eemd = EEMD(trials=50, noise_width=0.1)
    imfs = eemd.eemd(sig, t)
    reconstructed = np.sum(imfs[n_remove:], axis=0)
    return imfs, reconstructed

eemd_results = {}
eemd_imfs_all = {}
for name, (sig, t) in artifacts.items():
    imfs, recon = eemd_denoise(sig, t)
    eemd_imfs_all[name] = imfs
    eemd_results[name] = recon
    print(f'{name}: {imfs.shape[0]} IMFs extracted')

# Plot IMFs for each artifact
for name, (sig, t) in artifacts.items():
    imfs = eemd_imfs_all[name]
    n_imfs = min(imfs.shape[0], 8)
    fig, axes = plt.subplots(n_imfs + 1, 1, figsize=(14, 1.8 * (n_imfs + 1)), sharex=True)

    axes[0].plot(t, sig, color=colors[name], linewidth=0.5)
    axes[0].set_title(f'Original — {name}', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('µV')

    for i in range(n_imfs):
        f_psd, psd = welch(imfs[i], fs=fs, nperseg=min(512, len(imfs[i])))
        dom_freq = f_psd[np.argmax(psd)]
        energy_pct = np.sum(imfs[i]**2) / np.sum(sig**2) * 100
        axes[i+1].plot(t, imfs[i], color=plt.cm.tab10(i), linewidth=0.5)
        axes[i+1].set_title(f'IMF {i+1}  |  {dom_freq:.1f} Hz  |  {energy_pct:.1f}%',
                            fontsize=9)
        axes[i+1].set_ylabel('µV', fontsize=8)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'EEMD — {name}', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 6. CEEMDAN — Complete Ensemble EMD with Adaptive Noise
---
CEEMDAN adds adaptive noise at **each decomposition stage** (not just once), giving better reconstruction and less residual noise than EEMD.

In [ ]:
def ceemdan_denoise(sig, t, n_remove=1):
    """CEEMDAN decomposition, reconstruct by removing first n_remove IMFs."""
    ceemdan = CEEMDAN(trials=50, epsilon=0.1)
    imfs = ceemdan.ceemdan(sig, t)
    reconstructed = np.sum(imfs[n_remove:], axis=0)
    return imfs, reconstructed

ceemdan_results = {}
ceemdan_imfs_all = {}
for name, (sig, t) in artifacts.items():
    imfs, recon = ceemdan_denoise(sig, t)
    ceemdan_imfs_all[name] = imfs
    ceemdan_results[name] = recon
    print(f'{name}: {imfs.shape[0]} IMFs extracted')

# Plot IMFs for each artifact
for name, (sig, t) in artifacts.items():
    imfs = ceemdan_imfs_all[name]
    n_imfs = min(imfs.shape[0], 8)
    fig, axes = plt.subplots(n_imfs + 1, 1, figsize=(14, 1.8 * (n_imfs + 1)), sharex=True)

    axes[0].plot(t, sig, color=colors[name], linewidth=0.5)
    axes[0].set_title(f'Original — {name}', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('µV')

    for i in range(n_imfs):
        f_psd, psd = welch(imfs[i], fs=fs, nperseg=min(512, len(imfs[i])))
        dom_freq = f_psd[np.argmax(psd)]
        energy_pct = np.sum(imfs[i]**2) / np.sum(sig**2) * 100
        axes[i+1].plot(t, imfs[i], color=plt.cm.tab10(i), linewidth=0.5)
        axes[i+1].set_title(f'IMF {i+1}  |  {dom_freq:.1f} Hz  |  {energy_pct:.1f}%',
                            fontsize=9)
        axes[i+1].set_ylabel('µV', fontsize=8)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'CEEMDAN — {name}', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 7. VMD — Variational Mode Decomposition
---
VMD is optimization-based — we specify **K=5** modes targeting the main EEG bands. We reconstruct by removing mode 1 (highest center frequency, typically artifact/noise).

In [ ]:
K = 5
alpha = 2000
tau = 0
DC = 0
init = 1
tol = 1e-7

def vmd_denoise(sig, n_remove=1):
    """VMD decomposition, reconstruct by removing first n_remove modes."""
    modes, freqs, _ = VMD(sig, alpha, tau, K, DC, init, tol)
    # Sort modes by center frequency (ascending)
    order = np.argsort(freqs[-1])
    modes = modes[order]
    freqs_sorted = freqs[:, order]
    # Remove the highest-frequency mode(s)
    reconstructed = np.sum(modes[:-n_remove], axis=0)
    return modes, freqs_sorted, reconstructed

vmd_results = {}
vmd_modes_all = {}
vmd_freqs_all = {}
for name, (sig, t) in artifacts.items():
    modes, freqs, recon = vmd_denoise(sig)
    vmd_modes_all[name] = modes
    vmd_freqs_all[name] = freqs
    # Trim to match (VMD can drop 1 sample)
    vmd_results[name] = recon[:len(sig)] if len(recon) >= len(sig) else np.pad(recon, (0, len(sig)-len(recon)), mode='edge')
    print(f'{name}: {K} modes, center freqs = '
          f'{["{:.1f}".format(f*fs) for f in freqs[-1]]}')

# Plot modes for each artifact
for name, (sig, t) in artifacts.items():
    modes = vmd_modes_all[name]
    freqs = vmd_freqs_all[name]
    t_vmd = t[:modes.shape[1]]
    fig, axes = plt.subplots(K + 1, 1, figsize=(14, 1.8 * (K + 1)), sharex=True)

    axes[0].plot(t_vmd, sig[:len(t_vmd)], color=colors[name], linewidth=0.5)
    axes[0].set_title(f'Original — {name}', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('µV')

    colors_vmd = ['#e74c3c', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c']
    for i in range(K):
        f_psd, psd = welch(modes[i], fs=fs, nperseg=min(512, len(modes[i])))
        dom_freq = f_psd[np.argmax(psd)]
        energy_pct = np.sum(modes[i]**2) / np.sum(sig[:len(t_vmd)]**2) * 100
        axes[i+1].plot(t_vmd, modes[i], color=colors_vmd[i], linewidth=0.5)
        axes[i+1].set_title(
            f'Mode {i+1}  |  Center: {freqs[-1, i]*fs:.1f} Hz  |  '
            f'PSD peak: {dom_freq:.1f} Hz  |  {energy_pct:.1f}%',
            fontsize=9)
        axes[i+1].set_ylabel('µV', fontsize=8)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'VMD — {name} (K={K}, α={alpha})', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 8. Comparison — Original vs All Reconstructed Signals
---
For each artifact type, we overlay the **original** signal with the **reconstructed (denoised)** signal from all five methods. This lets us visually compare how aggressively each method removes the artifact and how much neural signal is preserved.

In [ ]:
method_colors = {
    'WT':      '#3498db',
    'EMD':     '#e67e22',
    'EEMD':    '#2ecc71',
    'CEEMDAN': '#9b59b6',
    'VMD':     '#e74c3c',
}

for name, (sig, t) in artifacts.items():
    fig, axes = plt.subplots(6, 1, figsize=(14, 14), sharex=True)

    # Original
    axes[0].plot(t, sig, color='gray', linewidth=0.5)
    axes[0].set_title(f'{name} — Original', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('µV')
    axes[0].grid(alpha=0.2)

    # Each method
    all_recons = {
        'WT': wt_results[name],
        'EMD': emd_results[name],
        'EEMD': eemd_results[name],
        'CEEMDAN': ceemdan_results[name],
        'VMD': vmd_results[name],
    }

    for ax, (method, recon) in zip(axes[1:], all_recons.items()):
        # Trim to common length
        n = min(len(sig), len(recon))
        ax.plot(t[:n], sig[:n], color='gray', linewidth=0.3, alpha=0.5, label='Original')
        ax.plot(t[:n], recon[:n], color=method_colors[method], linewidth=0.7,
                label=f'{method} reconstructed')
        ax.set_title(f'{method}', fontsize=11)
        ax.set_ylabel('µV')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(alpha=0.2)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'{name} — Original vs Reconstructed (All Methods)',
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 9. Quantitative Comparison
---
We compute the **energy removed** and **correlation with original** for each method on each artifact type. Higher energy removal suggests more aggressive artifact suppression; high correlation means the overall signal shape is preserved.

In [ ]:
print(f'{"Artifact":<18} {"Method":<10} {"Energy Removed (%)":>20} {"Correlation":>14}')
print('=' * 65)

all_methods = {
    'WT': wt_results,
    'EMD': emd_results,
    'EEMD': eemd_results,
    'CEEMDAN': ceemdan_results,
    'VMD': vmd_results,
}

for name, (sig, t) in artifacts.items():
    for method, results in all_methods.items():
        recon = results[name]
        n = min(len(sig), len(recon))
        orig, rec = sig[:n], recon[:n]
        energy_removed = (1 - np.sum(rec**2) / np.sum(orig**2)) * 100
        corr = np.corrcoef(orig, rec)[0, 1]
        print(f'{name:<18} {method:<10} {energy_removed:>18.1f}% {corr:>14.4f}')
    print('-' * 65)

---
### References
- [EEG Artifacts Handling — Brain Products](https://pressrelease.brainproducts.com/eeg-artifacts-handling-in-analyzer/)
- Huang et al., 1998 — The Empirical Mode Decomposition (EMD)
- Wu & Huang, 2009 — Ensemble EMD (EEMD)
- Torres et al., 2011 — CEEMDAN
- Dragomiretskiy & Zosso, 2014 — Variational Mode Decomposition (VMD)